In [1]:
import duckdb
import polars as pl
pl.Config.set_tbl_rows(60)

import pandas as pd

import shutil

In [2]:
data_path = '/media/dm/1TB/Data Fusion Contest 2026/task1/'
ver = "40"

In [3]:
conn = duckdb.connect('my_duckdb_database.db')

# Получение данных

In [4]:
%%time
#Консолидация данных с очисткой pretest.parquet от дублей

#если в планах нет использования порядка строк, оставить закомментированными строки с --file 

data = conn.sql(f"""
    SELECT * FROM
        (SELECT *, True AS pre, True AS train 
            FROM read_parquet([ '{data_path}pretrain_part_1.parquet',
                                '{data_path}pretrain_part_2.parquet',
                                '{data_path}pretrain_part_3.parquet'
                                ]
                            --file_row_number=True, filename=True 
                            )
        UNION ALL
        SELECT *, False AS pre, True AS train 
            FROM read_parquet([ '{data_path}train_part_1.parquet',
                                '{data_path}train_part_2.parquet',
                                '{data_path}train_part_3.parquet'
                                ]
                            --file_row_number=True, filename=True
                            )
        UNION ALL
        SELECT DISTINCT *, True AS pre, False AS train --удаление дублей в pretest.parquet
            FROM read_parquet('{data_path}pretest.parquet'
                                    --file_row_number=True, filename=True 
                                    ) 
                ANTI JOIN '{data_path}test.parquet' USING (event_id) --удаление пересечений с test.parquet
        UNION ALL
        SELECT *, False AS pre, False AS train 
            FROM read_parquet('{data_path}test.parquet'
                                --,file_row_number=True, filename=True
                                )
        ) ORDER BY event_dttm, event_id
        """)
#data.shape
#shape: (191_310_789, 27)
#(191310790, 25)
conn.sql(f"""
    DESCRIBE data
        """).df()

CPU times: user 27.7 ms, sys: 3.76 ms, total: 31.4 ms
Wall time: 29.5 ms


,column_name,column_type,null,key,default,extra
0,customer_id,BIGINT,YES,None,None,None
1,event_id,BIGINT,YES,None,None,None
2,event_dttm,VARCHAR,YES,None,None,None
3,event_type_nm,INTEGER,YES,None,None,None
4,event_desc,INTEGER,YES,None,None,None
5,channel_indicator_type,INTEGER,YES,None,None,None
6,channel_indicator_sub_type,INTEGER,YES,None,None,None
7,operaton_amt,DOUBLE,YES,None,None,None
8,currency_iso_cd,INTEGER,YES,None,None,None
9,mcc_code,VARCHAR,YES,None,None,None


In [5]:
%%time
#Типизация

#HASH используется если само значение неважно, но хочется подсчитывать использование этого значения ранее 
data = conn.sql(f"""
        SELECT 
            customer_id,  
            event_id,  
            event_dttm::TIMESTAMP AS event_dttm,
            --event_dttm::TIME AS time, -- лучше вычислять позже
            extract('hour' FROM event_dttm::TIMESTAMP)::UINT8 AS hour,
            ((extract('hour' FROM event_dttm::TIMESTAMP)+0)//2%12)::UINT8 AS hour2_0,
            ((extract('hour' FROM event_dttm::TIMESTAMP)+1)//2%12)::UINT8 AS hour2_1,
            
            event_type_nm::TINYINT AS event_type_nm,
            event_desc::SMALLINT AS event_desc,
            
            channel_indicator_type::TINYINT AS channel_indicator_type,
            channel_indicator_sub_type::TINYINT AS channel_indicator_sub_type,
            
            operaton_amt::FLOAT AS operaton_amt,
            
            COALESCE( currency_iso_cd, -1)::TINYINT AS currency_iso_cd,
            COALESCE( mcc_code, '-1')::TINYINT AS mcc_code,
            COALESCE( pos_cd, -1)::TINYINT AS pos_cd,
            
            (hash(COALESCE( accept_language,'-') )%(256*256-1))::UINT16 AS accept_language,--
            (hash(COALESCE( browser_language,'-') )%(256*256-1))::UINT16 AS browser_language,
            COALESCE( timezone, -1)::INT AS timezone,
            (session_id IS NULL) AS session_id_NULL,
            COALESCE( session_id, row_number() OVER (ORDER BY event_dttm, event_id) ) AS session_id,
            COALESCE( operating_system_type, 99)::TINYINT AS operating_system_type,
            --(battery IS NULL) AS battery,
            (hash(COALESCE( device_system_version,'-') )%(256*256-1))::UINT16 AS device_system_version,
            (hash(COALESCE( screen_size,'-') )%(256*256-1))::UINT16 AS screen_size,
            COALESCE( developer_tools,'9')::TINYINT AS developer_tools,
            COALESCE( phone_voip_call_state,'9')::TINYINT AS phone_voip_call_state,
            COALESCE( web_rdp_connection,'9')::TINYINT AS web_rdp_connection,
            COALESCE( compromised,'9')::TINYINT AS compromised,
            ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY event_dttm, event_id)::UINT16 AS CUST_ROW_NUMBER,
            pre,
            train,
            --file_row_number,
            --filename,
        FROM data
        ORDER BY customer_id, event_dttm, event_id
        """)

conn.sql(f"""
    DESCRIBE data
        """).df()

CPU times: user 33.7 ms, sys: 4.81 ms, total: 38.5 ms
Wall time: 34.4 ms


,column_name,column_type,null,key,default,extra
0,customer_id,BIGINT,YES,None,None,None
1,event_id,BIGINT,YES,None,None,None
2,event_dttm,TIMESTAMP,YES,None,None,None
3,hour,UTINYINT,YES,None,None,None
4,hour2_0,UTINYINT,YES,None,None,None
5,hour2_1,UTINYINT,YES,None,None,None
6,event_type_nm,TINYINT,YES,None,None,None
7,event_desc,SMALLINT,YES,None,None,None
8,channel_indicator_type,TINYINT,YES,None,None,None
9,channel_indicator_sub_type,TINYINT,YES,None,None,None


In [6]:
%%time
# запись консолидированного и типизированного в parquet
data.write_parquet(f'data{ver}.pq')
#Wall time: 9min 1s

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 48min 42s, sys: 7min 1s, total: 55min 43s
Wall time: 10min 42s


# просмотр parquet методами duckdb и polars

In [7]:
%%time
# просмотр данных методами duckdb и polars

#conn.sql(f""" SELECT * FROM read_parquet(f'data{ver}.pq') LIMIT 5 """).show()
#conn.sql(f""" SELECT * FROM read_parquet(f'data{ver}.pq') LIMIT 5 """).df()
pl.scan_parquet(f'data{ver}.pq').head().collect()
#pl.read_parquet(f'data{ver}.pq').head()

CPU times: user 111 ms, sys: 60.5 ms, total: 172 ms
Wall time: 188 ms


customer_id,event_id,event_dttm,hour,hour2_0,hour2_1,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,accept_language,browser_language,timezone,session_id_NULL,session_id,operating_system_type,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,CUST_ROW_NUMBER,pre,train
i64,i64,datetime[μs],u8,u8,u8,i8,i16,i8,i8,f32,i8,i8,i8,u16,u16,i32,bool,i64,i8,u16,u16,i8,i8,i8,i8,u16,bool,bool
123123123123129,123251972261925,2023-10-01 09:16:41,9,4,5,7,56,4,15,null,-1,-1,-1,24869,24869,-1,true,63410,99,24869,24869,9,9,9,9,1,true,true
123123123123129,125193298164858,2023-10-01 11:13:48,11,5,6,14,75,6,5,71085.0,0,15,8,24869,24869,-1,true,88323,99,24869,24869,9,9,9,9,2,true,true
123123123123129,124823932244729,2023-10-01 11:40:35,11,5,6,14,75,6,5,36158.0,0,14,8,24869,24869,-1,true,93658,99,24869,24869,9,9,9,9,3,true,true
123123123123129,124454563399321,2023-10-01 16:37:32,16,8,8,14,75,6,5,81971.0,0,4,8,24869,24869,-1,true,142943,99,24869,24869,9,9,9,9,4,true,true
123123123123129,126490377886229,2023-10-02 19:28:25,19,9,10,7,56,4,15,null,-1,-1,-1,24869,24869,-1,true,353428,99,24869,24869,9,9,9,9,5,true,true


# Примеры вычисления признаков

In [12]:
%%time
field = 'event_desc'

query_str=f"""
    SELECT *,

            --разница в секундах с предыдущей записью в рамках customer_id
            COALESCE( epoch(event_dttm - (LAG(event_dttm) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW))),-1)::INT32 AS LAG_CUST,            

            --АГРЕГАЦИИ в разрезе customer_id
            --количество записей ранее текущей записи в рамках customer_id
            COUNT(*) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::INT AS COUNT_BEFORE,            
            
            --количество записей с заполненым operaton_amt ранее текущей записи в рамках customer_id 
            COUNT(operaton_amt) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN  UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::INT AS SCOUNT_BEFORE,            
            
            --количество уникальных значений field ранее текущей записи в рамках customer_id
            --COUNT(DISTINCT {field}) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::INT AS UNIQUE_{field},            

            --сумма operaton_amt записей ранее текущей записи в рамках customer_id
            SUM(operaton_amt) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::FLOAT AS SUM_BEFORE,            
            
            --среднее operaton_amt записей ранее текущей записи в рамках customer_id
            MEAN(operaton_amt) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::FLOAT AS MEAN_BEFORE,            
            
            --медиана operaton_amt записей ранее текущей записи в рамках customer_id
            MEDIAN(operaton_amt) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::FLOAT AS MEDIAN_BEFORE,            
            
            --максимум operaton_amt записей ранее текущей записи в рамках customer_id
            MAX(operaton_amt) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::FLOAT AS MAX_BEFORE,            
            
            --максимум operaton_amt без 5% выбросов записей ранее текущей записи в рамках customer_id
            QUANTILE_CONT(operaton_amt, 0.95) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)::FLOAT AS MAX095_BEFORE,   


            --ПЕРИОДЫ
            --количество записей за последний час исключая текущую запись в рамках customer_id
            COUNT(*) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN INTERVAL 1 HOUR PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::INT AS COUNT_1H_,            

            --количество записей за последний час включая текущую запись в рамках customer_id
            COUNT(*) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN INTERVAL 1 HOUR PRECEDING AND CURRENT ROW)::INT AS COUNT_1H,            

            --количество записей до последнего часа в рамках customer_id
            COUNT(*) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 HOUR PRECEDING)::INT AS COUNT_BEFORE_1H,            

            --количество записей за месяц до последнего часа в рамках customer_id
            COUNT(*) OVER (PARTITION BY customer_id ORDER BY event_dttm RANGE BETWEEN INTERVAL 1 MONTH PRECEDING AND INTERVAL 1 HOUR PRECEDING)::INT AS COUNT_1M_1H,



            --ИМЕНОВАННЫЕ ОКНА
            MEDIAN(operaton_amt) OVER w_2M_1D::FLOAT AS MEDIAN_2M_1D,            
            SUM(operaton_amt) OVER w_2M_1D::FLOAT AS SUM_2M_1D,            
            QUANTILE_CONT(operaton_amt, 0.95) OVER w_2M_1D::FLOAT AS MAX095_2M_1D,            
            
            (
                (operaton_amt - AVG(operaton_amt) OVER w_2M_1D 
                )/stddev_samp(operaton_amt) OVER w_2M_1D 
            )::FLOAT AS ZScore_2M_1D,   



            -- в разрезе customer_id и field
            --разница в секундах с предыдущей записью в рамках customer_id и field
            COALESCE( epoch(event_dttm - (LAG(event_dttm) OVER (PARTITION BY customer_id, {field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW))),-1)::INT32 AS LAG_{field},            
            
            --количество записей с таким же значением field ранее текущей записи в рамках customer_id и field
            COUNT(*) OVER (PARTITION BY customer_id,{field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::INT AS COUNT_{field},            
            
            --количество записей с таким же значением field  с заполненым operaton_amt ранее текущей записи в рамках customer_id и field
            COUNT(operaton_amt) OVER (PARTITION BY customer_id,{field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::INT AS SCOUNT_{field},            

            --доля записей с таким же значением field ранее текущей записи в рамках customer_id
            (COUNT(*) OVER (PARTITION BY customer_id,{field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW))/CUST_ROW_NUMBER::FLOAT AS FRAC_{field},            
            
            --смягченная доля записей с таким же значением field ранее текущей записи в рамках customer_id и field
            (COUNT(*) OVER (PARTITION BY customer_id,{field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW))/(CUST_ROW_NUMBER+4)::FLOAT AS FRAC_{field},            


            --сумма operaton_amt записей с таким же значением field ранее текущей записи в рамках customer_id и field
            SUM(operaton_amt) OVER (PARTITION BY customer_id,{field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::FLOAT AS SUM_{field},            

            --среднее operaton_amt записей с таким же значением field ранее текущей записи в рамках customer_id и field
            AVG(operaton_amt) OVER (PARTITION BY customer_id,{field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::FLOAT AS AVG_{field},            

            --максиум operaton_amt записей с таким же значением field ранее текущей записи в рамках customer_id и field
            MAX(operaton_amt) OVER (PARTITION BY customer_id,{field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::FLOAT AS MAX_{field},            

            --медиана operaton_amt записей с таким же значением field ранее текущей записи в рамках customer_id и field
            MEDIAN(operaton_amt) OVER (PARTITION BY customer_id,{field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::FLOAT AS MEDIAN_{field},            

            --максимум operaton_amt без 5% выбросов записей с таким же значением field ранее текущей записи в рамках customer_id и field
            QUANTILE_CONT(operaton_amt, 0.95) OVER (PARTITION BY customer_id,{field} ORDER BY event_dttm RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW EXCLUDE CURRENT ROW)::FLOAT AS MAX095_{field},            
            
            
    FROM read_parquet('data{ver}.pq')
    
    WINDOW w_2M_1D AS (PARTITION BY customer_id 
                        ORDER BY event_dttm 
                        RANGE BETWEEN INTERVAL 2 MONTH PRECEDING AND INTERVAL 1 DAY PRECEDING)
            
        """
#print(query_str)
data = conn.sql(query_str)  
data.write_parquet(f'data{ver}a.pq')
#Wall time: 19min 55s

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 2h 25min 51s, sys: 2min 9s, total: 2h 28min
Wall time: 19min 55s


In [13]:
%%time
pl.scan_parquet(f'data{ver}a.pq').filter(pl.col("operaton_amt").is_not_null()).head(20).collect()

CPU times: user 38.7 s, sys: 3.05 s, total: 41.8 s
Wall time: 5.9 s


customer_id,event_id,event_dttm,hour,hour2_0,hour2_1,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,accept_language,browser_language,timezone,session_id_NULL,session_id,operating_system_type,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,CUST_ROW_NUMBER,pre,train,LAG_CUST,COUNT_BEFORE,SCOUNT_BEFORE,SUM_BEFORE,MEAN_BEFORE,MEDIAN_BEFORE,MAX_BEFORE,MAX095_BEFORE,COUNT_1H_,COUNT_1H,COUNT_BEFORE_1H,COUNT_1M_1H,MEDIAN_2M_1D,SUM_2M_1D,MAX095_2M_1D,ZScore_2M_1D,LAG_event_desc,COUNT_event_desc,SCOUNT_event_desc,FRAC_event_desc,FRAC_event_desc_1,SUM_event_desc,AVG_event_desc,MAX_event_desc,MEDIAN_event_desc,MAX095_event_desc
i64,i64,datetime[μs],u8,u8,u8,i8,i16,i8,i8,f32,i8,i8,i8,u16,u16,i32,bool,i64,i8,u16,u16,i8,i8,i8,i8,u16,bool,bool,i32,i32,i32,f32,f32,f32,f32,f32,i32,i32,i32,i32,f32,f32,f32,f32,i32,i32,i32,f32,f32,f32,f32,f32,f32,f32
123552619855103,126413069032651,2023-11-21 15:52:27,15,7,8,3,120,6,2,422226.0,0,10,8,24869,24869,-1,true,9998871,99,24869,24869,9,9,9,9,182,true,true,22,181,85,6.0145604e7,707595.375,25025.0,1.4972545e7,2.0169e6,3,4,178,110,25025.0,6.0145604e7,2.0184e6,-0.122755,-1,0,0,0.0,0.0,null,null,null,null,null
123552619855103,124068019062004,2024-03-01 08:20:57,8,4,4,3,120,6,2,1.2688202e7,0,10,3,24869,24869,-1,true,31858626,99,24869,24869,9,9,9,9,434,true,true,149,433,193,3.56919648e8,1.8493e6,37518.0,3.454868e7,1.2774e7,2,3,431,87,30165.5,1.4687307e7,1192331.5,27.073549,8699310,1,1,0.002304,0.002283,422226.0,422226.0,422226.0,422226.0,422226.0
123552619855103,123758781613185,2024-03-01 08:58:08,8,4,4,3,120,6,2,7.9584e6,0,10,3,24869,24869,-1,true,31872661,99,24869,24869,9,9,9,9,446,true,true,85,445,197,3.71272288e8,1.8846e6,37518.0,3.454868e7,1.2724973e7,14,15,431,87,30165.5,1.4687307e7,1192331.5,16.76022,2231,2,2,0.004484,0.004444,1.3110428e7,6.555214e6,1.2688202e7,6.555214e6,1.2074903e7
123552619855103,124196869248995,2025-06-25 16:26:07,16,8,8,3,120,0,5,6.34536e6,0,10,3,24869,24869,-1,true,185156356,99,24869,24869,9,9,9,9,1365,true,false,52,1364,412,9.98873536e8,2.4245e6,42585.0,8.3620592e7,1.1530591e7,12,13,1352,26,254013.5,3.48887392e8,8.3270192e7,-0.145112,41585279,3,3,0.002198,0.002191,2.1068828e7,7022942.5,1.2688202e7,7.9584e6,1.2215222e7
123552619855152,125115990040372,2023-10-02 06:31:15,6,3,3,14,15,4,15,19810.0,0,-1,-1,24869,24869,-1,true,201927,99,24869,24869,9,9,9,9,5,true,true,102,4,3,85760.0,28586.666016,29540.0,36284.0,35272.398438,1,2,3,3,null,null,null,null,-1,0,0,0.0,0.0,null,null,null,null,null
123552619855551,125382276529914,2023-10-02 04:21:12,4,2,2,14,75,6,5,29697.0,0,-1,19,24869,24869,-1,true,180551,99,24869,24869,9,9,9,9,3,true,true,53715,2,1,757125.0,757125.0,757125.0,757125.0,720753.625,0,1,2,2,null,null,null,null,-1,0,0,0.0,0.0,null,null,null,null,null
123552619855551,124609182821537,2023-10-09 07:24:24,7,3,4,14,75,6,5,39744.0,0,-1,19,24869,24869,-1,true,1459085,99,24869,24869,9,9,9,9,30,true,true,139392,29,19,1.6800994e7,884262.8125,901170.0,1.857112e6,1.8473e6,0,1,29,29,901170.0,1.6800994e7,1.8478e6,-1.981037,615792,1,1,0.033333,0.029412,29697.0,29697.0,29697.0,29697.0,29697.0
123552619855551,126155371036065,2023-10-10 11:24:39,11,5,6,14,75,6,5,39852.0,0,-1,19,24869,24869,-1,true,1710582,99,24869,24869,9,9,9,9,31,true,true,100815,30,20,1.6840738e7,842036.875,900135.0,1.857112e6,1.846808e6,0,1,30,30,900135.0,1.6840738e7,1.8473e6,-1.759632,100815,2,2,0.064516,0.057143,69441.0,34720.5,39744.0,34720.5,39241.648438
123552619855551,123243385659526,2023-10-11 15:21:37,15,7,8,14,75,6,5,39960.0,0,-1,19,24869,24869,-1,true,1978603,99,24869,24869,9,9,9,9,32,true,true,100618,31,21,1.688059e7,803837.625,899100.0,1.857112e6,1.8048e6,0,1,31,31,899100.0,1.688059e7,1.846808e6,-1.599485,100618,3,3,0.09375,0.083333,109293.0,36431.0,39852.0,39744.0,39841.199219


# Создание файлов TRAIN и TEST

In [16]:
%%time
#(наклейка меток из train_labels.parquet)
data = conn.sql(f"""
    SELECT 
            * EXCLUDE (target),
            (target IS NOT NULL)::BOOLEAN as label,
            COALESCE( target, 0)::BOOLEAN AS target
    FROM (SELECT 
            * EXCLUDE (pre, train),
        FROM read_parquet('data{ver}a.pq') 
        WHERE train AND NOT pre
    ) LEFT JOIN read_parquet('{data_path}train_labels.parquet') AS lbl USING (customer_id,event_id)
""")
data.write_parquet(f"train{ver}.pq")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 5min 6s, sys: 21.1 s, total: 5min 27s
Wall time: 45.2 s


In [19]:
%%time
data = conn.sql(f"""
        SELECT 
            * EXCLUDE (pre, train),
        FROM read_parquet('data{ver}a.pq') 
        WHERE NOT train AND NOT pre
""")
data.write_parquet("test{ver}.pq")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 1min 19s, sys: 6.73 s, total: 1min 25s
Wall time: 12.5 s
